# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [ ]:
"""The following generates the Quiz all our models will be evaluated on."""
import itertools
import string
import numpy as np

from dotenv import load_dotenv

from smolbench.induction.chromatic import (
	ChromaticIntervalsConfig,
	Prompter,
	get_random_exclusive_quiz,
	anneal_intervals
)
from smolbench.evals import (
	Marks
)
from smolbench.evals.openrouter import (
	evaluate
)

template = string.Template(
	"Context:\n"
	"---\n"
	"There is a ceremonial role called the $role, whose job it is to"
	" head the $parade parade. No one else besides the $role is able to head"
	" the $parade parade. The following lists the people who were $role and"
	" the years they were $role:\n"
	"$positive_info\n"
	"\n"
	"Query:\n"
	"Between the years $start and $end, exclusive of the end, could $color"
	" have headed the $parade parade every year? Answer with only one word:"
	" 'True' or 'False'."
)

def query_gen(
	labels_to_intervals: Dict[Color, Intervals],
	interval_to_label: Dict[Intervals, Color],
	seed: int,
) -> Dict[str, str]:
	"""Generates a series of queries"""
	rng: np.random.Generator = np.random.default_rng(seed)
	# Finds max interval.
	n: int = max(interval[1] for interval in interval_to_label)
	for color, intervals in labels_to_intervals.items():
		# Generates a series of true items.
		for start, end in intervals:
			start, end = np.sort(
				rng.choice(range(start, end + 1), size=2, replace=False)
			)
			yield {"color": color, "start": start, "end": end}, True
		# Generates a false proposition.
		invalid_range: Intervals = anneal_intervals(
			itertools.chain(
				(set(interval_to_label.keys()) - set(itertools.chain(*intervals)))
			)
		)
		for start, end in invalid_range:
			start = rng.choice(range(start, end))
			# Binom with p = intervals / n capped at end for a similar-ish
			# distr. to positive accounts.
			end = min(
				end,
				start
				+ rng.binomial(
					end - start + 1,
					np.mean([len(interval) for interval in interval_to_label]) / n,
				)
				+ 1,
			)
			yield {"color": color, "start": start, "end": end}, False


intens_quiz, extens_quiz = get_random_exclusive_quiz(
	ChromaticIntervalsConfig(
            n=250,
            intervals=250 // 4,
            colors=45,
            seed=1776,
        ),
	Prompter(
		template,
		{
			"role": "Twislax",
			"parade": "Gildane",
		},
		query_gen,
	),
)

# Decoder-Only Model
This section tests classical decoder-only models.

In [ ]:
decode_intens_eval: Marks = evaluate(intens_quiz, "mistralai/mistral-7b-instruct-v0.1")
decode_extens_eval: Marks = evaluate(extens_quiz, "mistralai/mistral-7b-instruct-v0.1")

In [ ]:
print(decode_extens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid)
print(decode_intens_eval.correct, decode_intens_eval.incorrect, decode_extens_eval.invalid)